# 🏀 I Analyzed 10 Years of NBA Data in Python — Here's What I Found

**Seasons covered:** 2014–15 through 2023–24  
**Tools:** Python · Pandas · Matplotlib · Seaborn  
**Data source:** Basketball Reference / NBA Stats  

---

## Table of Contents
1. [Setup & Data Loading](#1)
2. [The Three-Point Revolution](#2)
3. [Scoring Explosion: Is the NBA Too Soft?](#3)
4. [Who Dominated Scoring? (2014–2024)](#4)
5. [Dynasty Watch: Who Won the Most Rings?](#5)
6. [Player Prime Tracker: Curry, LeBron, Harden, Giannis](#6)
7. [True Shooting % — Who Was the Most Efficient?](#7)
8. [Key Findings & Takeaways](#8)

## 1. Setup & Data Loading <a id='1'></a>

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Global style ──────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#1a1d27',
    'axes.edgecolor':   '#2d3148',
    'axes.labelcolor':  '#e0e0e0',
    'xtick.color':      '#a0a0b0',
    'ytick.color':      '#a0a0b0',
    'text.color':       '#e0e0e0',
    'grid.color':       '#2d3148',
    'grid.linewidth':   0.6,
    'font.family':      'DejaVu Sans',
    'figure.dpi':       140,
})

NBA_BLUE   = '#1d428a'
NBA_RED    = '#c8102e'
NBA_GOLD   = '#ffc72c'
NBA_SILVER = '#8a8d90'
SEASON_COLORS = plt.cm.plasma(np.linspace(0.15, 0.85, 10))

print('✅ Libraries loaded. Ready to analyze 10 years of NBA data!')

In [ ]:
league  = pd.read_csv('data/league_stats.csv')
leaders = pd.read_csv('data/scoring_leaders.csv')
teams   = pd.read_csv('data/top_teams.csv')
careers = pd.read_csv('data/player_careers.csv')

print(f'League stats rows  : {len(league)}')
print(f'Scoring leaders    : {len(leaders)}')
print(f'Team records       : {len(teams)}')
print(f'Player career rows : {len(careers)}')
league.head()

## 2. The Three-Point Revolution 🌊 <a id='2'></a>

> *"The NBA changed. Here's the data to prove it."*

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('🌊 The Three-Point Revolution (2014–2024)',
             fontsize=16, fontweight='bold', color='white', y=1.01)

# ── Left: 3PA per game over time ──
ax = axes[0]
ax.set_facecolor('#1a1d27')
ax.plot(league['season'], league['3pa_per_game'],
        color=NBA_GOLD, linewidth=3, marker='o', markersize=8, zorder=5)
ax.fill_between(league['season'], league['3pa_per_game'],
                alpha=0.2, color=NBA_GOLD)

# Annotate start and end
ax.annotate(f"{league['3pa_per_game'].iloc[0]}/game",
            xy=(league['season'].iloc[0], league['3pa_per_game'].iloc[0]),
            xytext=(2015.3, 23.5), color=NBA_GOLD,
            fontsize=10, fontweight='bold')
ax.annotate(f"{league['3pa_per_game'].iloc[-1]}/game",
            xy=(league['season'].iloc[-1], league['3pa_per_game'].iloc[-1]),
            xytext=(2022.3, 35.5), color=NBA_GOLD,
            fontsize=10, fontweight='bold')

pct_increase = ((league['3pa_per_game'].iloc[-1] - league['3pa_per_game'].iloc[0])
                / league['3pa_per_game'].iloc[0] * 100)
ax.set_title(f'+{pct_increase:.0f}% increase in 3-point attempts',
             color='#a0a0b0', fontsize=11)
ax.set_xlabel('Season')
ax.set_ylabel('3-Point Attempts Per Game')
ax.set_xticks(league['season'])
ax.set_xticklabels(league['season_label'], rotation=45, ha='right', fontsize=8)
ax.grid(True, alpha=0.3)
ax.spines[['top','right']].set_visible(False)

# ── Right: Scoring vs pace correlation ──
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')
scatter = ax2.scatter(league['pace'], league['pts_per_game'],
                      c=league['season'], cmap='plasma',
                      s=120, zorder=5, edgecolors='white', linewidth=0.5)
for _, row in league.iterrows():
    ax2.annotate(str(row['season'])[-2:],
                 (row['pace'], row['pts_per_game']),
                 textcoords='offset points', xytext=(6, 4),
                 fontsize=8, color='#c0c0c0')

# Trend line
z = np.polyfit(league['pace'], league['pts_per_game'], 1)
p = np.poly1d(z)
x_line = np.linspace(league['pace'].min(), league['pace'].max(), 100)
ax2.plot(x_line, p(x_line), '--', color=NBA_RED, alpha=0.7, linewidth=2)

corr = league['pace'].corr(league['pts_per_game'])
ax2.set_title(f'Pace vs Scoring (r = {corr:.2f})',
              color='#a0a0b0', fontsize=11)
ax2.set_xlabel('Pace (possessions per 48 min)')
ax2.set_ylabel('Points Per Game (League Avg)')
ax2.grid(True, alpha=0.3)
ax2.spines[['top','right']].set_visible(False)
plt.colorbar(scatter, ax=ax2, label='Season')

plt.tight_layout()
plt.savefig('plots/three_point_revolution.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()
print(f'💡 Key Insight: 3PA per game jumped from {league["3pa_per_game"].iloc[0]} to '
      f'{league["3pa_per_game"].iloc[-1]} — a +{pct_increase:.0f}% increase in 10 years.')

## 3. Scoring Explosion: Is the NBA Too Soft? 🔥 <a id='3'></a>

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5.5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

bars = ax.bar(league['season_label'], league['pts_per_game'],
              color=SEASON_COLORS, edgecolor='#0f1117', linewidth=0.8, width=0.75)

# Add champion text inside bars
for bar, (_, row) in zip(bars, league.iterrows()):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
            f"{h:.1f}", ha='center', va='bottom',
            fontsize=9, fontweight='bold', color='white')
    champ_short = row['champion'].replace('Golden State ', 'GS ').replace(
        'Cleveland ', 'CLE ').replace('Toronto ', 'TOR ').replace(
        'Los Angeles ', 'LA ').replace('Milwaukee ', 'MIL ').replace(
        'Denver ', 'DEN ').replace('Boston ', 'BOS ')
    ax.text(bar.get_x() + bar.get_width()/2, 96,
            '🏆 ' + champ_short,
            ha='center', va='bottom',
            fontsize=6.5, color='#ffc72c', rotation=90)

ax.set_title('NBA League Average Points Per Game (2014–2024)',
             fontsize=15, fontweight='bold', color='white', pad=14)
ax.set_ylabel('Points Per Game')
ax.set_xlabel('Season')
ax.set_ylim(92, 118)
ax.tick_params(axis='x', rotation=45)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

total_increase = league['pts_per_game'].iloc[-1] - league['pts_per_game'].iloc[0]
ax.annotate(f'+{total_increase:.1f} pts/game over 10 years',
            xy=(7.5, 113.5), fontsize=11, color=NBA_GOLD,
            fontweight='bold',
            arrowprops=dict(arrowstyle='->', color=NBA_GOLD),
            xytext=(5.5, 116.5))

plt.tight_layout()
plt.savefig('plots/scoring_explosion.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 4. Who Dominated Scoring? (2014–2024) 🎯 <a id='4'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('🎯 NBA Scoring Leaders 2014–2024',
             fontsize=16, fontweight='bold', color='white')

# ── Left: PPG by season ──
ax = axes[0]
ax.set_facecolor('#1a1d27')

player_colors = {
    'Russell Westbrook': '#007AC1',
    'Stephen Curry':     '#FFC72C',
    'James Harden':      '#CE1141',
    'Joel Embiid':       '#006BB6',
    'Luka Doncic':       '#00538C',
}

bars = ax.barh(leaders['season_label'][::-1], leaders['ppg'][::-1],
               color=[player_colors.get(p, NBA_SILVER) for p in leaders['player'][::-1]],
               edgecolor='#0f1117', linewidth=0.6, height=0.65)

for bar, (_, row) in zip(bars, leaders[::-1].iterrows()):
    w = bar.get_width()
    ax.text(w + 0.2, bar.get_y() + bar.get_height()/2,
            f"{row['player']} ({row['ppg']})",
            va='center', fontsize=8, color='white')

ax.set_xlim(0, 46)
ax.set_xlabel('Points Per Game')
ax.set_title('PPG Scoring Leader Each Season', color='#a0a0b0', fontsize=11)
ax.grid(axis='x', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

legend_patches = [mpatches.Patch(color=c, label=p) for p, c in player_colors.items()]
ax.legend(handles=legend_patches, fontsize=8, loc='lower right',
          facecolor='#2a2d3a', edgecolor='#3d4060', labelcolor='white')

# ── Right: Wins above average for scoring leaders' teams ──
ax2 = axes[1]
ax2.set_facecolor('#1a1d27')

titles_count = leaders['player'].value_counts()
colors_pie   = [player_colors.get(p, NBA_SILVER) for p in titles_count.index]
wedges, texts, autotexts = ax2.pie(
    titles_count.values,
    labels=titles_count.index,
    colors=colors_pie,
    autopct='%1.0f%%',
    startangle=140,
    textprops={'color': 'white', 'fontsize': 9},
    wedgeprops={'edgecolor': '#0f1117', 'linewidth': 2},
    pctdistance=0.75
)
for at in autotexts:
    at.set_fontsize(10)
    at.set_fontweight('bold')

ax2.set_title('Share of Scoring Titles', color='#a0a0b0', fontsize=11)

plt.tight_layout()
plt.savefig('plots/scoring_leaders.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('\n📊 Scoring Title Leaders:')
for player, count in titles_count.items():
    print(f'  {player}: {count} scoring title(s)')

## 5. Dynasty Watch: Who Won the Most Rings? 💍 <a id='5'></a>

In [ ]:
champions = league['champion'].value_counts().reset_index()
champions.columns = ['team', 'championships']

team_colors = {
    'Golden State Warriors': '#FFC72C',
    'Cleveland Cavaliers':   '#860038',
    'Toronto Raptors':       '#CE1141',
    'Los Angeles Lakers':    '#552583',
    'Milwaukee Bucks':       '#00471B',
    'Denver Nuggets':        '#0E2240',
    'Boston Celtics':        '#007A33',
}

fig, ax = plt.subplots(figsize=(10, 5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

bars = ax.bar(champions['team'], champions['championships'],
              color=[team_colors.get(t, NBA_BLUE) for t in champions['team']],
              edgecolor='#0f1117', linewidth=0.8, width=0.6)

for bar, val in zip(bars, champions['championships']):
    rings = '💍' * val
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            rings + f'  ({val})',
            ha='center', va='bottom', fontsize=12)

ax.set_title('🏆 NBA Championships (2014–2024)',
             fontsize=15, fontweight='bold', color='white', pad=12)
ax.set_ylabel('Championships Won')
ax.set_ylim(0, 5.5)
ax.set_yticks(range(6))
ax.tick_params(axis='x', rotation=30)
ax.grid(axis='y', alpha=0.3)
ax.spines[['top','right']].set_visible(False)

plt.tight_layout()
plt.savefig('plots/championships.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()

gsw_pct = champions.loc[champions['team'] == 'Golden State Warriors', 'championships'].values[0] / 10 * 100
print(f'\n💡 Key Insight: Golden State Warriors won {gsw_pct:.0f}% of all championships this decade.')

## 6. Player Prime Tracker: Curry, LeBron, Harden, Giannis 📈 <a id='6'></a>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.patch.set_facecolor('#0f1117')
fig.suptitle('📈 Superstar Career Arcs (2014–2024)',
             fontsize=16, fontweight='bold', color='white', y=1.01)

players_cfg = [
    ('Stephen Curry',            '#FFC72C', axes[0, 0]),
    ('LeBron James',             '#CE1141', axes[0, 1]),
    ('James Harden',             '#CE1141', axes[1, 0]),
    ('Giannis Antetokounmpo',    '#00471B', axes[1, 1]),
]

for player, color, ax in players_cfg:
    df = careers[careers['player'] == player].sort_values('season')
    ax.set_facecolor('#1a1d27')

    ax.plot(df['season'], df['ppg'],
            color=color, linewidth=2.5, marker='o', markersize=7,
            label='PPG', zorder=5)
    ax.fill_between(df['season'], df['ppg'], alpha=0.15, color=color)

    ax2_twin = ax.twinx()
    ax2_twin.set_facecolor('#1a1d27')
    ax2_twin.plot(df['season'], df['ts_pct'],
                  color='#60a0ff', linewidth=1.8, marker='s', markersize=5,
                  linestyle='--', label='TS%', alpha=0.8)
    ax2_twin.set_ylabel('True Shooting %', color='#60a0ff', fontsize=9)
    ax2_twin.tick_params(axis='y', colors='#60a0ff')
    ax2_twin.spines[['top']].set_visible(False)
    ax2_twin.set_ylim(48, 72)

    peak_idx = df['ppg'].idxmax()
    ax.annotate(f"Peak: {df.loc[peak_idx, 'ppg']} PPG",
                xy=(df.loc[peak_idx, 'season'], df.loc[peak_idx, 'ppg']),
                xytext=(df.loc[peak_idx, 'season'] - 1, df.loc[peak_idx, 'ppg'] + 1.5),
                fontsize=8, color='white',
                arrowprops=dict(arrowstyle='->', color='white', lw=1))

    ax.set_title(player, color='white', fontsize=12, fontweight='bold')
    ax.set_xlabel('Season')
    ax.set_ylabel('Points Per Game', color=color)
    ax.tick_params(axis='y', colors=color)
    ax.set_xticks(df['season'])
    ax.set_xticklabels([str(s)[-2:] for s in df['season']],
                        fontsize=8)
    ax.grid(True, alpha=0.25)
    ax.spines[['top','right']].set_visible(False)

    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2_twin.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2,
              fontsize=8, facecolor='#2a2d3a',
              edgecolor='#3d4060', labelcolor='white',
              loc='lower left')

plt.tight_layout()
plt.savefig('plots/career_arcs.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()

## 7. True Shooting % — Who Was the Most Efficient? 🎯 <a id='7'></a>

In [ ]:
avg_stats = (careers
             .groupby('player')[['ppg', 'rpg', 'apg', 'ts_pct', '3pm']]
             .mean()
             .reset_index())
avg_stats = avg_stats.sort_values('ppg', ascending=False)

fig, ax = plt.subplots(figsize=(10, 5.5))
fig.patch.set_facecolor('#0f1117')
ax.set_facecolor('#1a1d27')

scatter_colors = ['#FFC72C', '#CE1141', '#552583', '#007A33', '#006BB6']
scatter = ax.scatter(avg_stats['ts_pct'], avg_stats['ppg'],
                     s=[v * 8 for v in avg_stats['3pm']],
                     c=scatter_colors[:len(avg_stats)],
                     alpha=0.9, edgecolors='white', linewidth=1.5, zorder=5)

for _, row in avg_stats.iterrows():
    name_short = row['player'].split()[-1]
    ax.annotate(name_short,
                (row['ts_pct'], row['ppg']),
                textcoords='offset points', xytext=(8, 5),
                fontsize=10, fontweight='bold', color='white')

ax.set_xlabel('True Shooting % (10-yr avg)', fontsize=11)
ax.set_ylabel('Points Per Game (10-yr avg)', fontsize=11)
ax.set_title('Scoring vs Efficiency — Bubble size = Avg 3PM/season',
             fontsize=12, fontweight='bold', color='white', pad=10)
ax.grid(True, alpha=0.25)
ax.spines[['top','right']].set_visible(False)

ax.axvline(avg_stats['ts_pct'].mean(), color=NBA_SILVER,
           linestyle='--', alpha=0.5, label='Avg TS%')
ax.legend(fontsize=9, facecolor='#2a2d3a', edgecolor='#3d4060', labelcolor='white')

plt.tight_layout()
plt.savefig('plots/efficiency_scatter.png', dpi=150,
            bbox_inches='tight', facecolor='#0f1117')
plt.show()

print('\n📊 10-Year Averages:')
print(avg_stats[['player','ppg','ts_pct','3pm']]
      .rename(columns={'ppg':'PPG','ts_pct':'TS%','3pm':'3PM/Season'})
      .to_string(index=False))

## 8. Key Findings & Takeaways 🧠 <a id='8'></a>

In [ ]:
league_start = league.iloc[0]
league_end   = league.iloc[-1]

print('=' * 65)
print('  🏀  10 YEARS OF NBA DATA — KEY FINDINGS')
print('=' * 65)

print(f'''
📌 FINDING 1 — The Three-Point Revolution Is Real
   3-point attempts per game: {league_start["3pa_per_game"]} → {league_end["3pa_per_game"]}
   That's a +{(league_end["3pa_per_game"]-league_start["3pa_per_game"])/league_start["3pa_per_game"]*100:.0f}% increase. Stephen Curry (2015–16: 402 threes)
   single-handedly rewired how teams play offense.

📌 FINDING 2 — Scoring Has Never Been Higher
   League PPG: {league_start["pts_per_game"]} → {league_end["pts_per_game"]} (+{league_end["pts_per_game"]-league_start["pts_per_game"]:.1f} pts/game)
   Pace + efficiency gains drove this — not just rule changes.

📌 FINDING 3 — James Harden's 2018-19 Was Historic
   36.1 PPG — the highest single-season average in 30+ years.
   He also had the highest ever 3-point attempts in a season (1,028).

📌 FINDING 4 — Golden State Warriors Defined the Decade
   4 Championships in 10 years (2015, 2017, 2018, 2022).
   Their 2015-16 season (73-9) is the best record in NBA history.

📌 FINDING 5 — LeBron James Is an Anomaly
   Still averaging 25+ PPG at age 39. No player in NBA history
   has sustained elite performance across 10+ years like this.

📌 FINDING 6 — True Shooting % Has Hit All-Time Highs
   TS% went from {league_start["ts_pct"]}% to {league_end["ts_pct"]}% — teams are shooting
   smarter, not just more. Corner threes and layups dominate.
''')
print('=' * 65)
print('  Analysis by Dr. Moussa Doumbia | github.com/doumbiassa')
print('=' * 65)

---
**Want to fetch live data instead?** Run `python fetch_live_data.py` first — it pulls real-time stats from the NBA API and overwrites the CSV files in `data/`.

```bash
pip install nba_api
python fetch_live_data.py
```